# Delivery times: service bands and a Pareto table

**Frequency Distribution and Histogram** · a worked example from the StatNex teaching guide

**The story.** The table holds the delivery times in minutes of 50 orders from each of three food-delivery apps: ZipEats, FeastNow and UrbanBite. Marketing wants a simple picture of "how long you usually wait" with each app.

**The question.** How are delivery times distributed for each app, and where do most orders fall?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `01_basic/frequency_histogram.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,ZipEats,FeastNow,UrbanBite
1.0,16.0,32.0,36.0
2.0,24.0,26.0,41.0
3.0,28.0,31.0,38.0
4.0,19.0,33.0,48.0
5.0,29.0,25.0,33.0
6.0,26.0,37.0,42.0
7.0,22.0,30.0,42.0
8.0,30.0,28.0,42.0
9.0,22.0,29.0,34.0
10.0,26.0,35.0,46.0
11.0,28.0,33.0,38.0
12.0,17.0,27.0,53.0
13.0,29.0,34.0,41.0
14.0,26.0,26.0,45.0
15.0,23.0,23.0,35.0
16.0,23.0,30.0,46.0
17.0,23.0,30.0,47.0
18.0,22.0,33.0,33.0
19.0,26.0,35.0,44.0
20.0,12.0,29.0,31.0
21.0,26.0,29.0,32.0
22.0,14.0,42.0,44.0
23.0,29.0,33.0,42.0
24.0,20.0,32.0,39.0
25.0,23.0,32.0,41.0
26.0,20.0,41.0,38.0
27.0,27.0,29.0,46.0
28.0,23.0,36.0,38.0
29.0,28.0,34.0,36.0
30.0,19.0,33.0,31.0
31.0,27.0,30.0,36.0
32.0,27.0,32.0,38.0
33.0,29.0,38.0,32.0
34.0,24.0,35.0,46.0
35.0,26.0,41.0,45.0
36.0,18.0,31.0,38.0
37.0,25.0,37.0,44.0
38.0,58.0,37.0,34.0
39.0,17.0,33.0,32.0
40.0,31.0,31.0,29.0
41.0,27.0,40.0,45.0
42.0,18.0,39.0,40.0
43.0,24.0,35.0,37.0
44.0,29.0,37.0,42.0
45.0,25.0,35.0,43.0
46.0,24.0,35.0,36.0
47.0,25.0,31.0,39.0
48.0,23.0,33.0,40.0
49.0,20.0,37.0,46.0
50.0,26.0,33.0,44.0

## Libraries

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per variable. Blank and text cells are left out, so columns may differ
# in length.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["ZipEats", "FeastNow", "UrbanBite"]
series = {c: pd.to_numeric(data[c], errors="coerce").dropna().to_numpy(float) for c in columns}

## Descriptive statistics

In [ ]:
# Variance and SD divide by n - 1; quartiles interpolate linearly between the sorted
# values (NumPy's default, Excel's QUARTILE.INC). The fences Q1 - 1.5 IQR and
# Q3 + 1.5 IQR mark where a value starts to count as an outlier.
def describe(x):
    q1, median, q3 = np.percentile(x, [25, 50, 75])
    mean, sd = np.mean(x), np.std(x, ddof=1)
    return {"n": x.size, "mean": mean, "variance": np.var(x, ddof=1), "sd": sd,
            "min": x.min(), "max": x.max(), "cv": sd / mean if mean else np.nan,
            "Q1 - 1.5 IQR": q1 - 1.5 * (q3 - q1), "Q1": q1, "median": median, "Q3": q3,
            "Q3 + 1.5 IQR": q3 + 1.5 * (q3 - q1)}


summary = pd.DataFrame({name: describe(x) for name, x in series.items()})
print(summary)

## Classes

In [ ]:
# The class boundaries as given, in ascending order. They need not be equally spaced.
boundaries = [float(b) for b in re.split(r"[,\s、]+", "0, 20, 30, 45, 60".strip()) if b]


def class_edges(x):
    return np.array(boundaries)

## Frequency tables

In [ ]:
# Classes are [lower, upper): a value on a boundary belongs to the class above, except
# that the last class also holds its upper bound.
frequencies, edges_of = {}, {}
for name, x in series.items():
    edges = class_edges(x)
    edges_of[name] = edges
    counts = np.histogram(x, bins=edges)[0]
    relative = counts / x.size
    frequencies[name] = pd.DataFrame({"lower": edges[:-1], "frequency": counts,
                                      "relative": relative, "cumulative": np.cumsum(relative)})
    print(f"\n{name}: class boundaries {np.round(edges, 6).tolist()}")
    print(frequencies[name].to_string(index=False))

## Table for a Pareto chart

In [ ]:
# The classes from the most frequent to the least; classes with the same frequency
# keep their order (a stable sort), and the running share is added down the table.
pareto_tables = {}
for name, table in frequencies.items():
    ordered = table.iloc[np.argsort(-table["frequency"].to_numpy(), kind="stable")]
    pareto_tables[name] = pd.DataFrame({"lower": ordered["lower"].to_numpy(),
                                        "frequency": ordered["frequency"].to_numpy(),
                                        "relative": ordered["relative"].to_numpy(),
                                        "cumulative": np.cumsum(ordered["relative"].to_numpy())})
    print(f"\n{name}")
    print(pareto_tables[name].to_string(index=False))

## Histograms

In [ ]:
Path("figures").mkdir(exist_ok=True)
fig, axes = plt.subplots(len(series), 1, figsize=(7, 3 * len(series)), squeeze=False)
for ax, (name, table) in zip(axes[:, 0], frequencies.items()):
    widths = np.diff(edges_of[name])
    # With unequal classes the area, not the height, carries the count: frequency density.
    unequal = not np.allclose(widths, widths[0])
    ax.bar(table["lower"], table["frequency"] / widths if unequal else table["frequency"],
           width=widths, align="edge", edgecolor="black")
    ax.set_title(name)
    ax.set_ylabel("Frequency per unit width" if unequal else "Frequency")
fig.tight_layout()
fig.savefig("figures/histogram.png", dpi=150)
plt.close(fig)

## What the result says

Here the classes are service bands chosen by hand: under 20 minutes, 20 to 30, 30 to 45, and 45 to 60. The classes are not equally wide, so compare the shares, not the heights of the bars.

For ZipEats, 38 of the 50 orders (relative frequency 0.76) arrive in 20 to 30 minutes and 9 arrive faster. For FeastNow, 40 orders (0.8) fall in 30 to 45 minutes. For UrbanBite, 38 orders fall in 30 to 45 minutes and 11 (0.22) take 45 minutes or more.

The table for a Pareto chart sorts the bands from the fullest to the emptiest and adds up the shares. For ZipEats the two fullest bands hold 0.94 of the orders.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.